<a href="https://colab.research.google.com/github/nikkk8/agentic-graph-rag-influencer-discovery/blob/main/Agentic_Graph_RAG_Over_Social_Network_Knowledge_Graphs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic Graph-RAG for Influencer Discovery

This notebook implements an end-to-end **Agentic Graph-RAG pipeline** for identifying influential nodes in a social network.

Pipeline Overview:

1. Graph Construction & Feature Engineering  
2. GCN-based Influence Ranking  
3. Subgraph Retrieval  
4. LLM-based Graph Reasoning (Open-source model)  
5. Influence Neighborhood Visualization  

The system demonstrates how graph analytics and LLM reasoning can be combined to perform influencer discovery in social networks.

Fully reproducible in Google Colab with open-source models.

In [ ]:
%pip -q install langgraph pydantic pandas networkx matplotlib tqdm scipy
%pip -q install torch==2.8.0+cpu torchvision==0.23.0+cpu torchaudio==2.8.0+cpu --index-url https://download.pytorch.org/whl/cpu
%pip -q install torch_geometric==2.6.1
!pip install -q transformers accelerate bitsandbytes

print("✅ Dependencies installed (you may restart the kernel if needed).")

In [ ]:
import os, json, random
from pathlib import Path
from typing import TypedDict, Dict, Any, Tuple, List

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.utils import from_networkx as pyg_from_nx, add_self_loops
from torch_geometric.nn import GCNConv
from transformers import AutoTokenizer, AutoModelForCausalLM

from langgraph.graph import StateGraph, START, END

# Reproducibility
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)

device = torch.device("cpu")  # keep CPU for reproducibility
plt.rcParams["figure.figsize"] = (8, 6)

print("✅ Imports complete | Using device:", device)


In [ ]:
USERS_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/GSNJkoEM3yeeCjJl1l2Jrg/users.csv"
EDGES_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/m9iBI6GCId0XoGEkjwHk3g/edges-follow.csv"

CUR = Path("data/curated"); CUR.mkdir(parents=True, exist_ok=True)
USERS_CSV = CUR / "users.csv"
EDGES_CSV = CUR / "edges_follow.csv"

# download only if missing
if not USERS_CSV.exists():
    pd.read_csv(USERS_URL).to_csv(USERS_CSV, index=False)
if not EDGES_CSV.exists():
    pd.read_csv(EDGES_URL).to_csv(EDGES_CSV, index=False)

df_users = pd.read_csv(USERS_CSV)
df_edges = pd.read_csv(EDGES_CSV)
display(df_users.head(3))
display(df_edges.head(3))
print(f"✅ users={len(df_users):,} | edges={len(df_edges):,}")


In [ ]:
# prompt = "Find influencers for politician and company pages"
# device = "cuda" if torch.cuda.is_available() else "cpu"

# inputs = tokenizer(prompt, return_tensors="pt").to(device)
# outputs = model.generate(
#     **inputs,
#     max_new_tokens=300,
#     do_sample=True,
#     temperature=0.7
# )

# print(tokenizer.decode(outputs[0], skip_special_tokens=True))


In [ ]:
def _topics_field(v):
    if isinstance(v, list): return [str(x).lower() for x in v]
    if isinstance(v, str) and v.startswith("["):
        try: return [str(x).lower() for x in json.loads(v)]
        except Exception: pass
    return [t.strip().lower() for t in str(v).split("|") if t.strip()]

df_users["topics"] = df_users["topics"].apply(_topics_field)
df_users["topics"].head()


## Graph Construction & Feature Engineering

In [ ]:
def build_graph(users_df: pd.DataFrame, edges_df: pd.DataFrame) -> Tuple[nx.DiGraph, pd.DataFrame]:
    G = nx.DiGraph()

    # Add nodes with attributes
    for _, r in users_df.iterrows():
        G.add_node(
            r["login"],
            **dict(
                name=r.get("name") or r["login"],
                company=str(r.get("company", "")),
                followers=int(r.get("followers", 0)),
                following=int(r.get("following", 0)),
                posts_30d=int(r.get("posts_30d", 0)),
                topics=r.get("topics", []),
                bio=str(r.get("bio", "")),
            )
        )

    # Add directed edges
    for _, e in edges_df.iterrows():
        s, d = e["src"], e["dst"]
        if s in G and d in G and s != d:
            G.add_edge(s, d, etype=e.get("etype","FOLLOW"))

    return G, users_df

G, users_df = build_graph(df_users, df_edges)

print(f"[graph] |V|={G.number_of_nodes()} |E|={G.number_of_edges()}")
print("Is DAG? ", nx.is_directed_acyclic_graph(G))
print("Weakly connected components:", nx.number_weakly_connected_components(G))


In [ ]:
def topic_vector(ts, vocab):
    v = np.zeros(len(vocab), dtype=np.float32)
    idx = {t:i for i,t in enumerate(vocab)}
    for t in ts:
        if t in idx: v[idx[t]] = 1.0
    return v

def cosine(a,b):
    na,nb = np.linalg.norm(a), np.linalg.norm(b)
    return 0.0 if na==0 or nb==0 else float(np.dot(a,b)/(na*nb))


In [ ]:
def make_features(G: nx.DiGraph, qtopics=None, normalize=True):
    vocab = sorted({t for _, d in G.nodes(data=True) for t in d.get("topics", [])})
    qv = topic_vector([t.lower() for t in (qtopics or [])], vocab)

    Gu = G.to_undirected()

    # Deterministic PageRank
    pr = {}
    if G.number_of_nodes():
        nstart = {n: 1.0 / G.number_of_nodes() for n in G}
        pr = nx.pagerank(G, nstart=nstart)

    deg_in, deg_out = dict(G.in_degree()), dict(G.out_degree())
    kcore = nx.core_number(Gu) if G.number_of_nodes() else {n: 0 for n in G}
    clust = nx.clustering(Gu) if G.number_of_nodes() else {n: 0.0 for n in G}

    X, nodes = [], []
    for n, d in G.nodes(data=True):
        nodes.append(n)
        tv = topic_vector(d.get("topics", []), vocab)
        topicality = cosine(tv, qv) if len(qv) else 0.0

        X.append([
            pr.get(n, 0.0),
            deg_in.get(n, 0),
            deg_out.get(n, 0),
            kcore.get(n, 0),
            clust.get(n, 0.0),
            d.get("posts_30d", 0),
            topicality,
        ])

    X = np.asarray(X, dtype=np.float32)
    if normalize and len(X):
        X = (X - X.mean(axis=0, keepdims=True)) / (X.std(axis=0, keepdims=True) + 1e-6)

    names = ["pagerank", "deg_in", "deg_out", "kcore", "clust", "posts_30d", "topic_sim"]
    return nodes, X, names


In [ ]:
def simple_target(G, nodes, X, names):
    idx = {f:i for i,f in enumerate(names)}
    s = (0.45*X[:,idx["pagerank"]] +
         0.25*(X[:,idx["deg_in"]] / (1+np.maximum(1.0, X[:,idx["deg_out"]]))) +
         0.15*X[:,idx["posts_30d"]] +
         0.15*X[:,idx["topic_sim"]])
    s = (s - s.min())/(1e-8 + (s.max()-s.min()))
    return s.astype(np.float32)


## Influence Ranking via Graph Convolutional Network

In [ ]:
class InfluenceGCN(nn.Module):
    def __init__(self, in_dim, hid=64, p_drop=0.1):
        super().__init__()
        self.gc1 = GCNConv(in_dim, hid, cached=False, add_self_loops=False)
        self.gc2 = GCNConv(hid, hid, cached=False, add_self_loops=False)
        self.out = nn.Linear(hid, 1)
        self.drop = nn.Dropout(p_drop)

    def forward(self, x, edge_index):
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
        h = self.gc1(x, edge_index)
        h = F.relu(h)
        h = self.drop(h)
        h = self.gc2(h, edge_index)
        h = F.relu(h)
        h = self.drop(h)
        return self.out(h).squeeze(-1)


In [ ]:
def train_ranker(G, qtopics, epochs=100, lr=2e-3, pairs_per_epoch=1024):
    # Build features and heuristic targets
    nodes, X, names = make_features(G, qtopics, normalize=True)
    y = simple_target(G, nodes, X, names)

    # Convert to PyG graph
    H = G.to_directed()
    d = pyg_from_nx(H)
    n2i = {n: i for i, n in enumerate(list(H.nodes()))}

    feat = np.zeros((len(n2i), X.shape[1]), dtype=np.float32)
    for i, n in enumerate(nodes):
        if n in n2i:
            feat[n2i[n]] = X[i]

    d.x = torch.tensor(feat, dtype=torch.float32, device=device)
    d.y = torch.tensor(y, dtype=torch.float32, device=device)
    d.edge_index = d.edge_index.to(device)

    model = InfluenceGCN(d.x.shape[1]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    # Pre-draw pairs using a deterministic generator
    g = torch.Generator(device="cpu").manual_seed(SEED)
    N = d.x.shape[0]
    idx_all = torch.randint(0, N, (epochs, pairs_per_epoch), generator=g)
    jdx_all = torch.randint(0, N, (epochs, pairs_per_epoch), generator=g)

    for ep in range(epochs):
        model.train()
        opt.zero_grad()

        s = model(d.x, d.edge_index)
        idx = idx_all[ep].to(device)
        jdx = jdx_all[ep].to(device)

        mask = (d.y[idx] > d.y[jdx])
        if mask.any():
            loss = -torch.log(torch.sigmoid(s[idx[mask]] - s[jdx[mask]])).mean()
            loss.backward()
            opt.step()
        else:
            loss = torch.tensor(0.0, device=device)

        if ep % 5 == 0:
            print(f"-train- ep {ep:02d} loss {loss.item():.4f}")

    model.eval()
    return model


In [ ]:
print("[gnn] training GCN ranker…")

trained_model = train_ranker(
    G,
    qtopics=["politician", "company", "tv_show", "news", "brand"],
    epochs=100,
    lr=2e-3,
)

print("✅ Training complete.")


## Graph-based Retrieval


In [ ]:
def link_entities(q: str):
    q = (q or "").lower()
    cand = ["politician", "company", "tv_show", "news", "brand", "community", "sports", "music"]
    topics = [t for t in cand if t in q] or ["politician", "company", "tv_show"]
    return {"topics": topics}


In [ ]:
def graph_retrieve(G, ents, k=2, top_n=800):
    topics = ents.get("topics", [])

    # Seeds: users whose topics intersect the query topics
    seeds = [n for n, d in G.nodes(data=True) if set(d.get("topics", [])) & set(topics)] \
            or list(G)[:200]

    S = set()
    for s in seeds[:100]:
        S.add(s)
        fringe = {s}
        for _ in range(k):
            nxt = set()
            for u in list(fringe):
                nxt |= set(G.successors(u))
                nxt |= set(G.predecessors(u))
            S |= nxt
            fringe = nxt

    return G.subgraph(sorted(list(S))[:top_n]).copy()


In [ ]:
def score_subgraph(Gsub, qtopics, model):
    nodes, X, names = make_features(Gsub, qtopics, normalize=True)

    H = Gsub.to_directed()
    d = pyg_from_nx(H)
    n2i = {n: i for i, n in enumerate(list(H.nodes()))}

    feat = np.zeros((len(n2i), X.shape[1]), dtype=np.float32)
    for i, n in enumerate(nodes):
        if n in n2i:
            feat[n2i[n]] = X[i]

    d.x = torch.tensor(feat, dtype=torch.float32, device=device)
    d.edge_index = d.edge_index.to(device)

    with torch.no_grad():
        s = trained_model(d.x, d.edge_index).detach().cpu().numpy()

    return {n: float(s[n2i[n]]) for n in nodes if n in n2i}


## LLM-based Graph Reasoning

In [ ]:
model_id = "microsoft/phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
)



In [ ]:
def llm_summarize(query, topk, Gsub):
    bullets = []
    for i, (u, sc, _) in enumerate(topk[:8], 1):
        d = Gsub.nodes[u]
        bullets.append(
            f"{i}. node={u} | score={sc:.2f} | "
            f"in={Gsub.in_degree(u)} out={Gsub.out_degree(u)} | "
            f"topics={','.join(d.get('topics', []))} | "
            f"posts_30d={d.get('posts_30d', 0)}"
        )

    # ---- Build a single instruction-style prompt ----
    prompt = f"""
You are a precise social-graph analyst.
Use ONLY the evidence provided.

Task:
1. Return a ranked list of 3–5 influencers
2. Give a one-sentence rationale for each
3. Provide a short outreach plan grounded in the graph structure

User query:
{query}

Evidence:
{chr(10).join(bullets)}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=500,
            do_sample=False,      # deterministic (like temperature=0)
            temperature=0.0
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()


In [ ]:
class AgentState(TypedDict, total=False):
    query: str
    plan: str
    entities: Dict[str, Any]
    subgraph: nx.DiGraph
    scores: Dict[str, float]
    answer: str


def _plan(s: AgentState) -> AgentState:
    s["plan"] = "rank_influencers_and_explain"
    s["entities"] = link_entities(s.get("query", ""))
    return s


def _retrieve(s: AgentState) -> AgentState:
    s["subgraph"] = graph_retrieve(G, s["entities"], k=2, top_n=800)
    return s


def _score(s: AgentState) -> AgentState:
    q = s.get("entities", {}).get("topics", [])
    s["scores"] = score_subgraph(s["subgraph"], q, model=trained_model)
    return s


def _synthesize(s: AgentState) -> AgentState:
    q = s.get("entities", {}).get("topics", [])
    nodes, X, names = make_features(s["subgraph"], q)

    # PageRank on the subgraph for extra explanation features
    pr = (
        nx.pagerank(
            s["subgraph"],
            nstart={n: 1 / max(1, s["subgraph"].number_of_nodes()) for n in s["subgraph"]},
        )
        if s["subgraph"].number_of_nodes()
        else {}
    )

    triples = []
    for u in nodes:
        sc = s["scores"].get(u, 0.0)
        d = s["subgraph"].nodes[u]
        why = (
            f"PR~{pr.get(u, 0):.2f}, "
            f"deg_in={s['subgraph'].in_degree(u)}, "
            f"posts={d.get('posts_30d', 0)}, "
            f"topics={','.join(d.get('topics', []))}"
        )
        triples.append((u, sc, why))

    triples.sort(key=lambda x: (-x[1], str(x[0])))
    topk = triples[:10]

    s["answer"] = llm_summarize(s.get("query", ""), topk, s["subgraph"])
    s["scores"] = {u: sc for u, sc, _ in topk}
    return s


graph = StateGraph(AgentState)
graph.add_node("plan", _plan)
graph.add_node("retrieve", _retrieve)
graph.add_node("score", _score)
graph.add_node("synthesize", _synthesize)

graph.add_edge(START, "plan")
graph.add_edge("plan", "retrieve")
graph.add_edge("retrieve", "score")
graph.add_edge("score", "synthesize")
graph.add_edge("synthesize", END)

app = graph.compile()

print("✅ LangGraph agent compiled.")


In [ ]:
def retrieve_topk(query, G, k=10):
    """
    Graph-based retriever.
    Scores nodes using degree + activity signals.
    """
    scored = []
    for u, data in G.nodes(data=True):
        score = 0.0

        # structural importance
        score += G.in_degree(u) * 1.0
        score += G.out_degree(u) * 0.5

        # activity signal
        score += data.get("posts_30d", 0) * 0.2

        # topical relevance (weak heuristic)
        topics = data.get("topics", [])
        if isinstance(topics, list):
            for t in topics:
                if t.lower() in query.lower():
                    score += 2.0

        scored.append((u, score, None))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:k]


## Agent Wrapper

In [ ]:
def agent_invoke(query):
    topk = retrieve_topk(query, G, k=10)
    nodes = [u for (u, _, _) in topk]
    Gsub = G.subgraph(nodes).copy()

    # build scores dict for visualization
    scores = {u: sc for (u, sc, _) in topk}

    return {
        "answer": llm_summarize(query, topk, Gsub),
        "scores": scores
    }


## Influence Neighborhood Visualization

In [ ]:
def plot_top3_clean(G, scores, hops=1, title="Top-3 influencer neighborhoods"):
    top = [u for u, _ in sorted(scores.items(), key=lambda x: (-x[1], str(x[0])))[:3]]

    H = nx.DiGraph()
    for c in top:
        if c not in G:
            continue
        H.add_node(c, **G.nodes[c])
        fringe = {c}
        for _ in range(hops):
            nxt = set()
            for u in list(fringe):
                for v in G.successors(u):
                    H.add_node(v, **G.nodes[v])
                    H.add_edge(u, v)
                    nxt.add(v)
                for v in G.predecessors(u):
                    H.add_node(v, **G.nodes[v])
                    H.add_edge(v, u)
                    nxt.add(v)
            fringe = nxt

    if H.number_of_nodes() == 0:
        print("[viz] empty")
        return

    pos = nx.spring_layout(H, seed=SEED)

    # Color top influencers differently
    node_colors = ["red" if n in top else "skyblue" for n in H.nodes()]
    node_sizes = [300 if n in top else 30 for n in H.nodes()]

    plt.figure(figsize=(10, 8))
    nx.draw_networkx_nodes(H, pos, node_color=node_colors, node_size=node_sizes, alpha=0.9)
    nx.draw_networkx_edges(H, pos, alpha=0.2)

    # Label ONLY top nodes
    labels = {n: n for n in top}
    nx.draw_networkx_labels(H, pos, labels=labels, font_size=10, font_color="black")

    plt.title(title)
    plt.axis("off")
    plt.show()


## Demo

The following example query asks the system to identify influential accounts related to political and company topics.

The agent retrieves a subgraph, ranks nodes using a GCN model, and produces a graph-grounded explanation using an open-source LLM.

In [ ]:
query = "Find influencers for politician and company pages"
res = agent_invoke(query)

print(res["answer"])
plot_top3_clean(G, res["scores"])
